# Experiment: TITLE

## Introduction
Describe context, motivation, expected impact, and the practical objective of this modeling study.

## Objectives
- Primary objective:
- Secondary objectives:
- Success criteria:
- Operational constraints:


## Index
- [Research framing](#Research-framing)
- [Setup and reproducibility](#Setup-and-reproducibility)
- [Data loading and overview](#Data-loading-and-overview)
- [EDA and representation analysis](#EDA-and-representation-analysis)
- [Data quality and leakage checks](#Data-quality-and-leakage-checks)
- [Split policy](#Split-policy)
- [Preprocessing and feature engineering](#Preprocessing-and-feature-engineering)
- [Model portfolio](#Model-portfolio)
- [Validation and metrics](#Validation-and-metrics)
- [Hyperparameter optimization](#Hyperparameter-optimization)
- [Holdout evaluation](#Holdout-evaluation)
- [Feature importance and interpretability](#Feature-importance-and-interpretability)
- [Error analysis and calibration](#Error-analysis-and-calibration)
- [Robustness and sensitivity](#Robustness-and-sensitivity)
- [Model artifact](#Model-artifact)
- [Limitations and threats to validity](#Limitations-and-threats-to-validity)
- [Reproducibility notes](#Reproducibility-notes)
- [Conclusion (to be written)](#Conclusion-(to-be-written))


## Research framing
- Task type (classification, regression, ranking, forecasting):
- Target variable and prediction horizon:
- Grouping variable for grouped validation (if any):
- Main hypotheses:
- Decision context and risk tolerance:


## Setup and reproducibility
Use a broad stack that covers preprocessing, feature selection, decomposition, manifold learning, pipelines, model families, advanced search, and robust evaluation.


In [ ]:
from __future__ import annotations

import os
import pickle
import random
import warnings
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from sklearn.experimental import enable_halving_search_cv  # noqa: F401

from sklearn.base import clone
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA, TruncatedSVD
from sklearn.dummy import DummyClassifier, DummyRegressor
from sklearn.ensemble import (
    ExtraTreesClassifier,
    ExtraTreesRegressor,
    GradientBoostingClassifier,
    GradientBoostingRegressor,
    HistGradientBoostingClassifier,
    HistGradientBoostingRegressor,
    RandomForestClassifier,
    RandomForestRegressor,
)
from sklearn.feature_selection import SelectFromModel, SelectKBest, f_classif, f_regression, mutual_info_classif, mutual_info_regression
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import ElasticNet, LogisticRegression, Ridge
from sklearn.manifold import TSNE
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    log_loss,
    mean_absolute_error,
    mean_squared_error,
    precision_score,
    r2_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import (
    GridSearchCV,
    GroupShuffleSplit,
    HalvingRandomSearchCV,
    KFold,
    StratifiedGroupKFold,
    StratifiedKFold,
    cross_val_predict,
    cross_validate,
    train_test_split,
)
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.neural_network import MLPClassifier, MLPRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, PolynomialFeatures, StandardScaler, label_binarize
from sklearn.svm import SVC, SVR
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor

try:
    from xgboost import XGBClassifier, XGBRegressor
    XGBOOST_AVAILABLE = True
except Exception:
    XGBOOST_AVAILABLE = False

warnings.filterwarnings("ignore")
plt.style.use("seaborn-v0_8-whitegrid")
sns.set_context("notebook")

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)

print("XGBoost available:", XGBOOST_AVAILABLE)


In [ ]:
CONFIG: dict[str, Any] = {
    "data_path": "path/to/data.csv",
    "target_col": "target",
    "group_col": None,
    "id_cols": [],
    "task_type": "classification",
    "test_size": 0.2,
    "cv_folds": 5,
    "n_iter_tuning": 20,
    "k_best": 50,
    "random_state": SEED,
    "save_plots": False,
    "models_dir": "models",
    "model_filename": "best_model.pkl",
}
CONFIG


## Data loading and overview
Document source, extraction date, labeling process, and possible bias.


In [ ]:
DATA_PATH = Path(CONFIG["data_path"])
if not DATA_PATH.exists():
    raise FileNotFoundError(f"Missing data file: {DATA_PATH}")

df = pd.read_csv(DATA_PATH)
target_col = CONFIG["target_col"]
if target_col not in df.columns:
    raise KeyError(f"Target column '{target_col}' not found")

print("Dataset preview before cleaning:")
display(df.head(10))
display(df.head())
print("Shape:", df.shape)
display(df[target_col].value_counts(dropna=False).rename("target_count").to_frame())


## EDA and representation analysis
Include descriptive EDA plus dimensionality diagnostics with PCA and t-SNE.


In [ ]:
profile = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing_pct": df.isna().mean().mul(100),
    "n_unique": df.nunique(dropna=True),
}).sort_values(by="missing_pct", ascending=False)
display(profile.head(30))

num_cols = [c for c in df.select_dtypes(include=["number"]).columns if c != target_col]
if len(num_cols) >= 2:
    plt.figure(figsize=(9, 7))
    sns.heatmap(df[num_cols[:15]].corr(), cmap="coolwarm", center=0)
    plt.title("Correlation heatmap (subset)")
    plt.tight_layout()
    if CONFIG.get("save_plots", False):
        Path("plots").mkdir(parents=True, exist_ok=True)
        plt.savefig("plots/correlation_heatmap.png", dpi=200, bbox_inches="tight")


In [ ]:
# PCA and t-SNE visualization scaffold
num_cols = [c for c in df.select_dtypes(include=["number"]).columns if c != target_col]
if len(num_cols) >= 3:
    sample_df = df[num_cols + [target_col]].dropna().copy()
    if len(sample_df) > 3000:
        sample_df = sample_df.sample(3000, random_state=SEED)

    X_num = sample_df[num_cols].to_numpy()
    y_vis = sample_df[target_col].to_numpy()

    scaler_vis = StandardScaler()
    X_scaled = scaler_vis.fit_transform(X_num)

    pca_2d = PCA(n_components=2, random_state=SEED)
    X_pca = pca_2d.fit_transform(X_scaled)

    tsne_2d = TSNE(n_components=2, init="pca", learning_rate="auto", perplexity=30, random_state=SEED)
    X_tsne = tsne_2d.fit_transform(X_scaled)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].scatter(X_pca[:, 0], X_pca[:, 1], c=pd.factorize(y_vis)[0], s=12, alpha=0.7)
    axes[0].set_title("PCA projection")
    axes[0].set_xlabel("PC1")
    axes[0].set_ylabel("PC2")

    axes[1].scatter(X_tsne[:, 0], X_tsne[:, 1], c=pd.factorize(y_vis)[0], s=12, alpha=0.7)
    axes[1].set_title("t-SNE projection")
    axes[1].set_xlabel("t-SNE 1")
    axes[1].set_ylabel("t-SNE 2")
    plt.tight_layout()
    if CONFIG.get("save_plots", False):
        Path("plots").mkdir(parents=True, exist_ok=True)
        plt.savefig("plots/pca_tsne_projection.png", dpi=200, bbox_inches="tight")


## Data quality and leakage checks
Include missingness, duplicates, outliers, and leakage risk screening.


In [ ]:
missing_pct = df.isna().mean().mul(100).sort_values(ascending=False)
duplicates = int(df.duplicated().sum())
print("Duplicate rows:", duplicates)
display(missing_pct.head(30).rename("missing_pct").to_frame())

num_for_outlier = [c for c in df.select_dtypes(include=["number"]).columns if c != target_col]
if num_for_outlier:
    z = (df[num_for_outlier] - df[num_for_outlier].mean()) / (df[num_for_outlier].std(ddof=0) + 1e-12)
    outlier_rate = (z.abs() > 3).mean().sort_values(ascending=False)
    display(outlier_rate.head(20).rename("outlier_rate_abs_z_gt_3").to_frame())

# Minimal cleaning scaffold (customize as needed)
df_clean = df.drop_duplicates().copy()
print("Dataset shape before cleaning:", df.shape)
print("Dataset shape after cleaning:", df_clean.shape)
print("Dataset preview after cleaning:")
display(df_clean.head(10))
df = df_clean


## Split policy
Use train_test_split for default holdout, GroupShuffleSplit when grouping is required.


In [ ]:
drop_cols = list(CONFIG.get("id_cols", [])) + [target_col]
drop_cols = [c for c in drop_cols if c in df.columns]
X = df.drop(columns=drop_cols)
y = df[target_col]
groups = None
group_col = CONFIG.get("group_col")
if group_col and group_col in df.columns:
    groups = df[group_col]

if groups is not None:
    gss = GroupShuffleSplit(n_splits=1, test_size=CONFIG["test_size"], random_state=SEED)
    train_idx, test_idx = next(gss.split(X, y, groups=groups))
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
    groups_train = groups.iloc[train_idx]
else:
    stratify = y if CONFIG["task_type"] == "classification" else None
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=CONFIG["test_size"], random_state=SEED, stratify=stratify
    )
    groups_train = None

print("Train:", X_train.shape, "Test:", X_test.shape)

is_classification = CONFIG["task_type"] == "classification"
is_multiclass = False
class_labels = None
positive_label = CONFIG.get("positive_label", 1)
primary_metric_cls = "roc_auc"

if is_classification:
    class_labels = np.sort(pd.unique(y_train))
    n_classes = int(len(class_labels))
    is_multiclass = n_classes > 2

    if is_multiclass:
        primary_metric_cls = "roc_auc_ovr_weighted"
    else:
        if positive_label not in class_labels:
            positive_label = class_labels[-1]

    print("Classification detected. Classes:", class_labels.tolist())
    print("Multiclass:", is_multiclass)
    if not is_multiclass:
        print("Positive label used for binary metrics:", positive_label)


## Preprocessing and feature engineering
Use preprocessing, polynomial expansion, and feature selection in a leakage-safe pipeline.


In [ ]:
num_features = X_train.select_dtypes(include=["number"]).columns.tolist()
cat_features = X_train.select_dtypes(exclude=["number"]).columns.tolist()

numeric_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("poly", PolynomialFeatures(degree=2, include_bias=False)),
])

categorical_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipe, num_features),
        ("cat", categorical_pipe, cat_features),
    ],
    remainder="drop",
)

if CONFIG["task_type"] == "classification":
    univariate_selector = SelectKBest(score_func=mutual_info_classif, k=CONFIG["k_best"])
    fallback_selector = SelectKBest(score_func=f_classif, k=CONFIG["k_best"])
else:
    univariate_selector = SelectKBest(score_func=mutual_info_regression, k=CONFIG["k_best"])
    fallback_selector = SelectKBest(score_func=f_regression, k=CONFIG["k_best"])

print("Numeric features:", len(num_features), "Categorical features:", len(cat_features))


## Model portfolio
Include broad families from sklearn and optionally xgboost.


In [ ]:
if CONFIG["task_type"] == "classification":
    models: dict[str, Any] = {
        "baseline": DummyClassifier(strategy="most_frequent"),
        "logreg": LogisticRegression(max_iter=3000, random_state=SEED),
        "svc": SVC(probability=True, random_state=SEED),
        "knn": KNeighborsClassifier(),
        "tree": DecisionTreeClassifier(random_state=SEED),
        "rf": RandomForestClassifier(random_state=SEED),
        "extra_trees": ExtraTreesClassifier(random_state=SEED),
        "gb": GradientBoostingClassifier(random_state=SEED),
        "hgb": HistGradientBoostingClassifier(random_state=SEED),
        "mlp": MLPClassifier(hidden_layer_sizes=(128, 64), max_iter=300, random_state=SEED),
    }
    if XGBOOST_AVAILABLE:
        models["xgboost"] = XGBClassifier(
            random_state=SEED,
            n_estimators=300,
            max_depth=6,
            learning_rate=0.05,
            subsample=0.9,
            colsample_bytree=0.9,
            eval_metric="logloss",
            n_jobs=-1,
        )
else:
    models = {
        "baseline": DummyRegressor(strategy="mean"),
        "ridge": Ridge(random_state=SEED),
        "elastic_net": ElasticNet(random_state=SEED),
        "svr": SVR(),
        "knn": KNeighborsRegressor(),
        "tree": DecisionTreeRegressor(random_state=SEED),
        "rf": RandomForestRegressor(random_state=SEED),
        "extra_trees": ExtraTreesRegressor(random_state=SEED),
        "gb": GradientBoostingRegressor(random_state=SEED),
        "hgb": HistGradientBoostingRegressor(random_state=SEED),
        "mlp": MLPRegressor(hidden_layer_sizes=(128, 64), max_iter=300, random_state=SEED),
    }
    if XGBOOST_AVAILABLE:
        models["xgboost"] = XGBRegressor(
            random_state=SEED,
            n_estimators=300,
            max_depth=6,
            learning_rate=0.05,
            subsample=0.9,
            colsample_bytree=0.9,
            n_jobs=-1,
        )

list(models.keys())


## Validation and metrics
Use cross_validate for robust model comparison and cross_val_predict for out-of-fold diagnostics.


In [ ]:
if CONFIG["task_type"] == "classification":
    if groups_train is not None:
        cv = StratifiedGroupKFold(n_splits=CONFIG["cv_folds"], shuffle=True, random_state=SEED)
    else:
        cv = StratifiedKFold(n_splits=CONFIG["cv_folds"], shuffle=True, random_state=SEED)

    if is_multiclass:
        scoring = {
            "accuracy": "accuracy",
            "balanced_accuracy": "balanced_accuracy",
            "precision_weighted": "precision_weighted",
            "recall_weighted": "recall_weighted",
            "f1_weighted": "f1_weighted",
            "f1_macro": "f1_macro",
            "roc_auc_ovr_weighted": "roc_auc_ovr_weighted",
            "neg_log_loss": "neg_log_loss",
        }
    else:
        scoring = {
            "accuracy": "accuracy",
            "balanced_accuracy": "balanced_accuracy",
            "precision": "precision",
            "recall": "recall",
            "f1": "f1",
            "roc_auc": "roc_auc",
            "pr_auc": "average_precision",
            "neg_log_loss": "neg_log_loss",
        }
else:
    cv = KFold(n_splits=CONFIG["cv_folds"], shuffle=True, random_state=SEED)
    scoring = {
        "mae": "neg_mean_absolute_error",
        "mse": "neg_mean_squared_error",
        "rmse": "neg_root_mean_squared_error",
        "r2": "r2",
    }

rows: list[dict[str, Any]] = []
for name, estimator in models.items():
    pipe = Pipeline(steps=[
        ("prep", preprocessor),
        ("svd", TruncatedSVD(n_components=min(200, max(2, CONFIG["k_best"])))),
        ("select", univariate_selector),
        ("model", estimator),
    ])

    try:
        out = cross_validate(pipe, X_train, y_train, cv=cv, scoring=scoring, groups=groups_train, n_jobs=-1, return_train_score=False)
    except Exception:
        pipe = Pipeline(steps=[
            ("prep", preprocessor),
            ("select", fallback_selector),
            ("model", estimator),
        ])
        out = cross_validate(pipe, X_train, y_train, cv=cv, scoring=scoring, groups=groups_train, n_jobs=-1, return_train_score=False)

    row = {"model": name}
    for metric in scoring:
        vals = out[f"test_{metric}"]
        if metric.startswith("neg_"):
            metric_name = metric.replace("neg_", "")
            vals = -vals
        else:
            metric_name = metric
        row[f"{metric_name}_mean"] = float(np.mean(vals))
        row[f"{metric_name}_std"] = float(np.std(vals))
    rows.append(row)

cv_results = pd.DataFrame(rows)
display(cv_results)


In [ ]:
# Out-of-fold predictions for diagnostics
diag_name = cv_results.iloc[0]["model"] if not cv_results.empty else list(models.keys())[0]
diag_est = models[diag_name]
diag_pipe = Pipeline(steps=[("prep", preprocessor), ("select", fallback_selector), ("model", diag_est)])

if CONFIG["task_type"] == "classification":
    oof_pred = cross_val_predict(diag_pipe, X_train, y_train, cv=cv, groups=groups_train, method="predict", n_jobs=-1)
    print("OOF classification report for", diag_name)
    print(classification_report(y_train, oof_pred, zero_division=0))
else:
    oof_pred = cross_val_predict(diag_pipe, X_train, y_train, cv=cv, groups=groups_train, method="predict", n_jobs=-1)
    mse_oof = mean_squared_error(y_train, oof_pred)
    print("OOF RMSE for", diag_name, ":", float(np.sqrt(mse_oof)))


## Hyperparameter optimization
Use GridSearchCV and HalvingRandomSearchCV on selected strong candidates.


In [ ]:
top_models = cv_results.head(min(3, len(cv_results)))["model"].tolist() if not cv_results.empty else []
tuned_models: dict[str, Any] = {}
tuning_log: list[dict[str, Any]] = []

for name in top_models:
    base = models[name]
    pipe = Pipeline(steps=[("prep", preprocessor), ("select", fallback_selector), ("model", base)])

    if CONFIG["task_type"] == "classification":
        param_grid = {
            "model__max_depth": [None, 5, 10] if "tree" in name or "rf" in name or "xgboost" in name else [None],
        }
        refit_metric = primary_metric_cls if primary_metric_cls in scoring else list(scoring.keys())[0]
    else:
        param_grid = {
            "model__max_depth": [None, 5, 10] if "tree" in name or "rf" in name or "xgboost" in name else [None],
        }
        refit_metric = "rmse" if "rmse" in scoring else list(scoring.keys())[0]

    grid = GridSearchCV(pipe, param_grid=param_grid, scoring=scoring, refit=refit_metric, cv=cv, n_jobs=-1)
    grid.fit(X_train, y_train, **({"groups": groups_train} if groups_train is not None else {}))

    halving = HalvingRandomSearchCV(
        estimator=pipe,
        param_distributions=param_grid,
        scoring=scoring,
        refit=refit_metric,
        cv=cv,
        n_jobs=-1,
        random_state=SEED,
        factor=2,
        min_resources="smallest",
    )
    halving.fit(X_train, y_train, **({"groups": groups_train} if groups_train is not None else {}))

    best_search = grid if grid.best_score_ >= halving.best_score_ else halving
    tuned_models[name] = best_search.best_estimator_
    tuning_log.append({
        "model": name,
        "grid_best": float(grid.best_score_),
        "halving_best": float(halving.best_score_),
        "selected": "grid" if best_search is grid else "halving",
    })

display(pd.DataFrame(tuning_log))


## Holdout evaluation
Evaluate tuned models on untouched holdout data with a complete metric panel.


In [ ]:
holdout_rows: list[dict[str, Any]] = []

for name, est in tuned_models.items():
    est.fit(X_train, y_train)
    pred = est.predict(X_test)
    row = {"model": name}

    if CONFIG["task_type"] == "classification":
        row["accuracy"] = float(accuracy_score(y_test, pred))
        row["balanced_accuracy"] = float(balanced_accuracy_score(y_test, pred))

        if is_multiclass:
            row["precision_weighted"] = float(precision_score(y_test, pred, average="weighted", zero_division=0))
            row["recall_weighted"] = float(recall_score(y_test, pred, average="weighted", zero_division=0))
            row["f1_weighted"] = float(f1_score(y_test, pred, average="weighted", zero_division=0))
            row["f1_macro"] = float(f1_score(y_test, pred, average="macro", zero_division=0))

            if hasattr(est, "predict_proba"):
                prob = est.predict_proba(X_test)
                row["roc_auc_ovr_weighted"] = float(roc_auc_score(y_test, prob, multi_class="ovr", average="weighted"))
                row["log_loss"] = float(log_loss(y_test, prob, labels=class_labels))
                y_bin = label_binarize(y_test, classes=class_labels)
                row["pr_auc_weighted"] = float(average_precision_score(y_bin, prob, average="weighted"))
        else:
            row["precision"] = float(precision_score(y_test, pred, pos_label=positive_label, zero_division=0))
            row["recall"] = float(recall_score(y_test, pred, pos_label=positive_label, zero_division=0))
            row["f1"] = float(f1_score(y_test, pred, pos_label=positive_label, zero_division=0))

            if hasattr(est, "predict_proba"):
                prob_all = est.predict_proba(X_test)
                pos_idx = int(np.where(class_labels == positive_label)[0][0])
                prob = prob_all[:, pos_idx]
                row["roc_auc"] = float(roc_auc_score(y_test, prob))
                row["pr_auc"] = float(average_precision_score((y_test == positive_label).astype(int), prob))
                row["log_loss"] = float(log_loss(y_test, prob_all, labels=class_labels))
    else:
        mse = float(mean_squared_error(y_test, pred))
        row["mae"] = float(mean_absolute_error(y_test, pred))
        row["mse"] = mse
        row["rmse"] = float(np.sqrt(mse))
        row["r2"] = float(r2_score(y_test, pred))

    holdout_rows.append(row)

holdout_results = pd.DataFrame(holdout_rows)
display(holdout_results)


## Feature importance and interpretability
Use permutation importance and inspect top drivers for plausibility and leakage risks.


In [ ]:
if holdout_results.empty:
    raise ValueError("No holdout results available")

if CONFIG["task_type"] == "classification" and "roc_auc_ovr_weighted" in holdout_results.columns:
    best_name = holdout_results.sort_values(by="roc_auc_ovr_weighted", ascending=False).iloc[0]["model"]
elif CONFIG["task_type"] == "classification" and "roc_auc" in holdout_results.columns:
    best_name = holdout_results.sort_values(by="roc_auc", ascending=False).iloc[0]["model"]
elif CONFIG["task_type"] != "classification" and "rmse" in holdout_results.columns:
    best_name = holdout_results.sort_values(by="rmse", ascending=True).iloc[0]["model"]
else:
    best_name = holdout_results.iloc[0]["model"]

best_model = tuned_models[best_name]
best_model.fit(X_train, y_train)
perm = permutation_importance(best_model, X_test, y_test, n_repeats=15, random_state=SEED, n_jobs=-1)

feat_names = np.array([f"feature_{i}" for i in range(len(perm.importances_mean))])
fi = pd.DataFrame({
    "feature": feat_names,
    "importance_mean": perm.importances_mean,
    "importance_std": perm.importances_std,
}).sort_values(by="importance_mean", ascending=False)
display(fi.head(20))


## Error analysis and calibration
Use confusion analysis for classification, residual analysis for regression, and calibration diagnostics when probabilities are available.


In [ ]:
final_model = best_model
pred = final_model.predict(X_test)

if CONFIG["task_type"] == "classification":
    print("Confusion matrix")
    print(confusion_matrix(y_test, pred))
    print(classification_report(y_test, pred, zero_division=0))

    if hasattr(final_model, "predict_proba"):
        raw_prob_all = final_model.predict_proba(X_test)

        cal_model = CalibratedClassifierCV(estimator=clone(final_model), method="sigmoid", cv=3)
        cal_model.fit(X_train, y_train)
        cal_prob_all = cal_model.predict_proba(X_test)

        if is_multiclass:
            classes_to_plot = class_labels[: min(3, len(class_labels))]
            fig, axes = plt.subplots(1, len(classes_to_plot), figsize=(6 * len(classes_to_plot), 5))
            axes = np.array(axes).reshape(-1)
            y_test_arr = np.asarray(y_test)

            for ax, cls in zip(axes, classes_to_plot):
                cls_idx = int(np.where(class_labels == cls)[0][0])
                y_true_bin = (y_test_arr == cls).astype(int)
                frac_raw, mean_raw = calibration_curve(y_true_bin, raw_prob_all[:, cls_idx], n_bins=10, strategy="uniform")
                frac_cal, mean_cal = calibration_curve(y_true_bin, cal_prob_all[:, cls_idx], n_bins=10, strategy="uniform")

                ax.plot(mean_raw, frac_raw, marker="o", label="Uncalibrated")
                ax.plot(mean_cal, frac_cal, marker="o", label="Calibrated")
                ax.plot([0, 1], [0, 1], linestyle="--", label="Ideal")
                ax.set_title(f"Calibration (class={cls})")
                ax.set_xlabel("Mean predicted probability")
                ax.set_ylabel("Observed positive rate")
                ax.legend()
            plt.tight_layout()
            if CONFIG.get("save_plots", False):
                Path("plots").mkdir(parents=True, exist_ok=True)
                plt.savefig("plots/calibration_multiclass.png", dpi=200, bbox_inches="tight")
        else:
            pos_idx = int(np.where(class_labels == positive_label)[0][0])
            raw_prob = raw_prob_all[:, pos_idx]
            cal_prob = cal_prob_all[:, pos_idx]
            y_true_bin = (np.asarray(y_test) == positive_label).astype(int)

            frac_raw, mean_raw = calibration_curve(y_true_bin, raw_prob, n_bins=10, strategy="uniform")
            frac_cal, mean_cal = calibration_curve(y_true_bin, cal_prob, n_bins=10, strategy="uniform")

            plt.figure(figsize=(6, 6))
            plt.plot(mean_raw, frac_raw, marker="o", label="Uncalibrated")
            plt.plot(mean_cal, frac_cal, marker="o", label="Calibrated")
            plt.plot([0, 1], [0, 1], linestyle="--", label="Ideal")
            plt.xlabel("Mean predicted probability")
            plt.ylabel("Observed positive rate")
            plt.title("Calibration curve")
            plt.legend()
            plt.tight_layout()
            if CONFIG.get("save_plots", False):
                Path("plots").mkdir(parents=True, exist_ok=True)
                plt.savefig("plots/calibration_binary.png", dpi=200, bbox_inches="tight")
else:
    residual = y_test - pred
    plt.figure(figsize=(8, 5))
    sns.scatterplot(x=pred, y=residual)
    plt.axhline(0, linestyle="--", color="red")
    plt.xlabel("Predicted")
    plt.ylabel("Residual")
    plt.title("Residual diagnostics")
    plt.tight_layout()
    if CONFIG.get("save_plots", False):
        Path("plots").mkdir(parents=True, exist_ok=True)
        plt.savefig("plots/residual_diagnostics.png", dpi=200, bbox_inches="tight")


## Robustness and sensitivity
Assess seed sensitivity and consistency of model rankings under repeated validation.


In [ ]:
seed_grid = [7, 21, 42, 84, 168]
stability_rows: list[dict[str, Any]] = []

for s in seed_grid:
    if CONFIG["task_type"] == "classification":
        cv_s = StratifiedKFold(n_splits=CONFIG["cv_folds"], shuffle=True, random_state=s)
    else:
        cv_s = KFold(n_splits=CONFIG["cv_folds"], shuffle=True, random_state=s)

    out = cross_validate(clone(final_model), X_train, y_train, cv=cv_s, scoring=scoring, n_jobs=-1, return_train_score=False)
    row = {"seed": s}
    for metric in scoring:
        vals = out[f"test_{metric}"]
        key = metric.replace("neg_", "")
        if metric.startswith("neg_"):
            vals = -vals
        row[f"{key}_mean"] = float(np.mean(vals))
    stability_rows.append(row)

stability_df = pd.DataFrame(stability_rows)
display(stability_df)


## Model artifact
Persist the final selected model as a pickle file in `models/`.


In [ ]:
models_dir = Path(CONFIG.get("models_dir", "models"))
models_dir.mkdir(parents=True, exist_ok=True)
artifact_path = models_dir / CONFIG.get("model_filename", "best_model.pkl")

with artifact_path.open("wb") as f:
    pickle.dump(final_model, f)

print(f"Saved model artifact to: {artifact_path.resolve()}")


## Limitations and threats to validity
- Data limitations:
- Potential biases and representativeness gaps:
- Model assumptions and failure modes:
- External validity constraints:


## Reproducibility notes
- Environment and package versions:
- Random seeds and deterministic settings:
- Hardware and runtime notes:
- Full rerun instructions:


In [ ]:
# Optional environment capture
# import sys
# import sklearn
# print(sys.version)
# print("sklearn:", sklearn.__version__)
# print("numpy:", np.__version__)
# print("pandas:", pd.__version__)


## Conclusion (to be written)
Prompt suggestions only:
- Main findings and strongest evidence:
- Preferred model and rationale:
- Main risks and limitations:
- Recommended next experiments:
